# KORF Automation – Step-by-Step

**Rules:**

- KORF has a **5-open limit** – we NEVER launch a new instance.
- Every cell connects to the **already-running** KORF process.
- Run each cell one at a time and verify before moving on.


## Step 1 – Connect to the running KORF instance

This connects to the existing KORF process (it does **not** open a new one).
Make sure KORF is already open before running this cell.


In [1]:
import time
import os
from pywinauto.application import Application

KORF_PATH = r"C:\Program Files (x86)\Korf 36\Korf_36.exe"

# Connect to the EXISTING Korf instance – does NOT open a new one
app = Application(backend="win32").connect(path=KORF_PATH)
main_dlg = app.top_window()
main_dlg.set_focus()

print(f"Connected!  Window title: '{main_dlg.window_text()}'")
print(f"Window class: '{main_dlg.class_name()}'")

Connected!  Window title: 'KORF - [Pumpcases.kdf]'
Window class: 'ThunderRT6FormDC'


c:\Users\PrasannaPalanivel\OneDrive - CC7\Documents\Code\pyKorf\.venv\Lib\site-packages\pywinauto\application.py:1085: UserWarning: 32-bit application should be automated using 32-bit Python (you use 64-bit Python)
  warnings.warn(


## Step 2 – Inspect available menus & controls

Let's dump the menu bar items so we know the exact names to use.


In [2]:
# Print all top-level menu items
menu_bar = main_dlg.menu()
print("=== Top-level menu items ===")
for i, item in enumerate(menu_bar.items()):
    print(f"  [{i}] {item.text()}")

=== Top-level menu items ===
  [0] &File
  [1] &Edit
  [2] &View
  [3] Hy&draulics
  [4] &Process
  [5] &Tools
  [6] &Help


## Step 3 – Expand the "Hydraulics" menu & list sub-items

We drill into the Hydraulics menu to see the exact sub-menu structure.


In [5]:
# Find the Hydraulics top-level menu and list its children
menu_bar = main_dlg.menu()
for item in menu_bar.items():
    if "Hy&draulics" in item.text():
        print(f"Found top-level menu: '{item.text()}'")
        sub_menu = item.sub_menu()
        if sub_menu:
            print("  Sub-items:")
            for j, sub in enumerate(sub_menu.items()):
                print(f"    [{j}] '{sub.text()}'")
                # Check for a second-level submenu
                try:
                    sub_sub = sub.sub_menu()
                    if sub_sub:
                        for k, ss in enumerate(sub_sub.items()):
                            print(f"        [{j}.{k}] '{ss.text()}'")
                except Exception:
                    pass
        break
else:
    print("Hydraulics menu not found – check Step 2 output for the correct name.")

Found top-level menu: 'Hy&draulics'
  Sub-items:
    [0] '&Title'
    [1] ''
    [2] '&Cases'
    [3] ''
    [4] '&Specifications'
    [5] ''
    [6] '&Hydraulics'
        [6.0] '&Run'
        [6.1] 'R&esume'
        [6.2] ''
        [6.3] '&Stop'
    [7] ''
    [8] '&Results'
        [8.0] '&View Report'
        [8.1] '&Save Report'
        [8.2] ''
        [8.3] 'View &RunLog'
        [8.4] 'Save Run&Log'
        [8.5] ''
        [8.6] 'View E&xcel Report'


## Step 4 – Click Hydraulics → Hydraulics → Run

Confirmed menu structure from Step 3:

| Level     | Raw text      | Resolved       |
| --------- | ------------- | -------------- |
| Top-level | `Hy&draulics` | **Hydraulics** |
| Sub-menu  | `&Hydraulics` | **Hydraulics** |
| Command   | `&Run`        | **Run**        |

`pywinauto` strips `&` during matching, so `"Hydraulics -> Hydraulics -> Run"` is the correct selector.


In [8]:
# Re-acquire the top window and set focus (safe guard)
main_dlg = app.top_window()
main_dlg.set_focus()
time.sleep(0.5)

# Confirmed path from Step 3 inspection:
#   Top-level : 'Hy&draulics'  → Hydraulics
#   Sub-menu  : '&Hydraulics'  → Hydraulics
#   Command   : '&Run'         → Run
# pywinauto strips & during matching, so the path below is correct.
print("Navigating: Hydraulics → Hydraulics → Run ...")
main_dlg.menu_select("Hydraulics -> Hydraulics -> Run")
print("Run command sent successfully!")

Navigating: Hydraulics → Hydraulics → Run ...
Run command sent successfully!


## Step 5 – Handle the Runlog popup (if any)

After the run completes, KORF may show a "Runlog" dialog. This cell waits for it and clicks OK.


In [9]:
# Wait for KORF to finish calculating (Runlog popup confirms completion)
print("Waiting for Runlog dialog (up to 30 s)...")
try:
    runlog = app.window(title_re=".*[Rr]unlog.*")  # title may vary slightly
    runlog.wait("visible", timeout=30)
    print(f"Runlog appeared: '{runlog.window_text()}' – clicking OK.")
    runlog.OK.click()
    print("Runlog dismissed.")
except Exception as e:
    print(f"Runlog did not appear (run may have been instant or already closed): {e}")

Waiting for Runlog dialog (up to 30 s)...
Runlog appeared: 'Runlog - Case 3 solution reached after 3 iterations.' – clicking OK.
Runlog dismissed.


## Step 6 – Save the model (Ctrl+S)

Saves the results back so we can parse the output file.


In [ ]:
main_dlg = app.top_window()
main_dlg.set_focus()
main_dlg.type_keys("^s")
time.sleep(1)
print("Save command sent.")